In [8]:
import cme.decision_models.confidence_accumulation as ca
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import jax


In [3]:
n_states, start_width, threshold, measurement_prob, delta, mu, sigma, I, J = 7, 3, 1, 1, 1, np.asarray([[2]]), np.asarray([[1]]), 1, 5
model_type = "Markov"

In [4]:
intensity_matrix = ca.get_intensity_matrix(n_states, mu, sigma, model_type=model_type)
phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Centered")
intensity_matrix, phi_0

(Array([[[[-1. , -0.5,  0. ,  0. ,  0. ,  0. ,  0. ],
          [ 1. , -1. , -0.5,  0. ,  0. ,  0. ,  0. ],
          [ 0. ,  1.5, -1. , -0.5,  0. ,  0. ,  0. ],
          [ 0. ,  0. ,  1.5, -1. , -0.5,  0. ,  0. ],
          [ 0. ,  0. ,  0. ,  1.5, -1. , -0.5,  0. ],
          [ 0. ,  0. ,  0. ,  0. ,  1.5, -1. ,  1. ],
          [ 0. ,  0. ,  0. ,  0. ,  0. ,  1.5, -1. ]]]], dtype=float64),
 Array([[[[0.        ],
          [0.        ],
          [0.33333333],
          [0.33333333],
          [0.33333333],
          [0.        ],
          [0.        ]]]], dtype=float64))

In [5]:
Mc, Mw, Mn = ca._get_measurement_matrix(n_states, threshold, prob=measurement_prob, model_type = model_type)
Mc

Array([[0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1.]], dtype=float64)

In [33]:

t = np.asarray([[4.5]]) #t_s #
phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=t, RA_s = None, delta=delta, 
Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
transition_type="RT", likelihood_type="SINGLE")

phi_t.size

7

In [32]:
jax.jacfwd(ca.perform_state_transition)(intensity_matrix, t, None, delta, 
Mc, Mn, Mw, phi_0, 
"RT", "SINGLE").size

2401

In [31]:
jax.jacrev(ca.perform_state_transition)(intensity_matrix, t, None, delta, 
Mc, Mn, Mw, phi_0, 
"RT", "SINGLE").size

2401

In [30]:
jax.jacfwd(jax.jacrev(ca.perform_state_transition))(intensity_matrix, t, None, delta, 
Mc, Mn, Mw, phi_0, 
"RT", "SINGLE").size

117649

In [35]:
ca.likelihood(intensity_matrix, phi_0, delta, t, np.asarray([[1]]),Mc, Mw, Mn,"TIMESTEP","SINGLE", model_type)

Array([[0.74478421]], dtype=float64)